# Compare the WW3 outputs between different MCW configurations

In [ ]:
import xarray as xr
import cf_xarray
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client


import matplotlib.path as mpath
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.feature as cft
import cftime

# Stats
import geopandas as gpd
from scipy.interpolate import griddata

# Plotting
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import cmocean.cm as cmo

import os
import requests
import zipfile


from tqdm.notebook import tqdm 
import calendar
import pandas as pd
from datetime import datetime

# CLEANUP
%matplotlib inline
import seaborn as sns
import calendar


import xarray as xr
import cf_xarray
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client
import cmocean.cm as cmo

import matplotlib.path as mpath
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.feature as cft
import cftime

import geopandas as gpd

from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib as mpl
from scipy.interpolate import griddata

import os
import requests
import zipfile


from tqdm.notebook import tqdm 
import calendar
import pandas as pd
from datetime import datetime
import cartopy.feature as cfeature
from matplotlib.colors import LogNorm

blue_marble = plt.imread('/g/data/ik11/grids/BlueMarble.tiff')
blue_marble_extent = (-180, 180, -90, 90)


In [ ]:
client = Client(threads_per_worker=1)
client

In [ ]:
from typing import Optional, Sequence, Tuple, Dict, Any
def get_lon_lat_from_catalog(
    datastore,
    lon_candidates=("geolon", "geolon_t", "lonh", "lonq", "xt_ocean", "lon"),
    lat_candidates=("geolat", "geolat_t", "lath", "latq", "yt_ocean", "lat"),
) -> Tuple[xr.DataArray, xr.DataArray]:
    """
    Try to discover longitude and latitude fields from an intake-esm datastore.

    Parameters
    ----------
    datastore : intake_esm.core.esm_datastore
        The opened ESM datastore.
    lon_candidates, lat_candidates : tuple of str
        Candidate variable names to try in the catalog.

    Returns
    -------
    (lon, lat) : (xarray.DataArray, xarray.DataArray)
        Longitude and latitude arrays suitable for assigning as CF coords.
    """
    def _find_var(candidates):
        for name in candidates:
            try:
                cat = datastore.search(variable=name)
                if len(cat.df) == 0:
                    continue
                ds = cat.to_dask(
                    xarray_open_kwargs=dict(
                        chunks={},
                        decode_timedelta=True,
                        use_cftime=True,
                    ),
                    xarray_combine_by_coords_kwargs=dict(
                        compat="override",
                        data_vars="minimal",
                        coords="minimal",
                    ),
                )
                if name in ds:
                    da = ds[name]
                else:
                    # sometimes the DataArray is exposed as an attribute
                    da = getattr(ds, name)
                # drop any trivial time-like dims if present
                for d in ("time", "nv", "time_counter"):
                    if d in da.dims and da.sizes.get(d, 1) == 1:
                        da = da.isel({d: 0}, drop=True)
                return da
            except Exception:
                continue
        raise RuntimeError(f"Could not find any of {candidates} in datastore")

    lon = _find_var(lon_candidates)
    lat = _find_var(lat_candidates)

    return lon, lat

def _align_lonlat_dims(da_model, lon, lat):
    # propose mapping from common grid dims -> model dims
    proposed = {
        "lonh": "nx", "xh": "nx", "x": "nx", "xt_ocean": "nx", "lon": "nx", "longitude": "nx",
        "lath": "ny", "yh": "ny", "y": "ny", "yt_ocean": "ny", "lat": "ny", "latitude": "ny",
    }

    def _rename_only_existing(da):
        # keep only keys that exist in this DataArray (dims or coords)
        mapping = {k: v for k, v in proposed.items() if (k in da.dims) or (k in da.coords)}
        return da.rename(mapping) if mapping else da

    lon = _rename_only_existing(lon)
    lat = _rename_only_existing(lat)

    # sanity checks (only for overlapping dims)
    for d in set(lon.dims).intersection(da_model.dims):
        if lon.sizes[d] != da_model.sizes[d]:
            raise ValueError(f"lon size mismatch on dim {d}: {lon.sizes[d]} vs {da_model.sizes[d]}")
    for d in set(lat.dims).intersection(da_model.dims):
        if lat.sizes[d] != da_model.sizes[d]:
            raise ValueError(f"lat size mismatch on dim {d}: {lat.sizes[d]} vs {da_model.sizes[d]}")

    return lon, lat

In [ ]:
### USER EDIT start
# esm_file = "/g/data/ps29/nd0349/access-om3/configurations/mcw/IC4M8-MCW-100km_jra_iaf_2010/archive/experiment_datastore.json"
# esm_file2 = "/scratch/ps29/nd0349/access-om3/archive/MCW-100km_jra_iaf-2010-icdr/experiment_datastore.json"
# esm_file3 = "/scratch/ps29/nd0349/access-om3/archive/MCW-100km_jra_iaf-2010-icdr-weld//experiment_datastore.json"
# esm_file4 = "/scratch/ps29/nd0349/access-om3/archive/MCW-100km_jra_iaf-2010-icdr-test2/experiment_datastore.json"

esm_files = [
    "/g/data/ps29/nd0349/access-om3/configurations/mcw/IC4M8-MCW-100km_jra_iaf_2010/archive/experiment_datastore.json",
    "/scratch/ps29/nd0349/access-om3/archive/MCW-100km_jra_iaf-2010-icdr/experiment_datastore.json",
    "/scratch/ps29/nd0349/access-om3/archive/MCW-100km_jra_iaf-2010-icdr-weld/experiment_datastore.json",
    "/scratch/ps29/nd0349/access-om3/archive/MCW-100km_jra_iaf-2010-icdr-weld-IC4M8/experiment_datastore.json",
]
dpi=300
### USER EDIT stop

import os
from matplotlib import rcParams
%matplotlib inline
rcParams["figure.dpi"]= dpi

plotfolder=f"/g/data/{os.environ['PROJECT']}/{os.environ['USER']}/access-om3-paper-figs/"
os.makedirs(plotfolder, exist_ok=True)

 # a similar cell under this means it's being run in batch
for i in range(len(esm_files)):
    print("ESM datastore path:", esm_files[i])
print("Plot folder path: ",plotfolder)

In [ ]:
import os

def get_experiment_name(path):
    name = os.path.basename(os.path.dirname(os.path.dirname(path)))
    if name == "archive":
        name = os.path.basename(os.path.dirname(path))
    return name

expt_names = [get_experiment_name(f) for f in esm_files]
for i in range(len(expt_names)):
    print("Experiment name:", expt_names[i])

In [ ]:
def available_variables(datastore):
    """Return a pandas dataframe summarising the variables in a datastore"""
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastores = {}

for esm_file in esm_files:
    name = get_experiment_name(esm_file)
    
    datastore = intake.open_esm_datastore(
        esm_file,
        columns_with_iterables=COLUMNS_WITH_ITERABLES
    )
    
    ds_filtered = datastore.search(
        realm="seaIce",
        frequency="1day"
    )
    
    print(f"\n{name}")
    # available_variables(ds_filtered)
    
    datastores[name] = datastore

In [ ]:
datastore_filtered = datastores[expt_names[0]].search(realm="seaIce", frequency="1day")
available_variables(datastore_filtered)

In [ ]:
datastores[expt_names[0]].search(frequency="fx").unique()['variable']

In [ ]:
datastores.keys()

In [ ]:
expt_names

In [ ]:
def read_in_data(datastore):
    grid_lonlat_file = "/g/data/vk83/prerelease/configurations/inputs/access-om3/mom/initial_conditions/global.100km/2025.10.24/woa23_ts_01_mom.nc"
    grid_lon_name = "lon"
    grid_lat_name = "lat"
    # print("Reading in... ", datastore.keys())

    ds = datastore.search(variable=["aice", "wave_sig_ht", "fsdrad", "hi", "afsd", "dafsd_wave", "dafsd_weld"],
                      frequency="1day", 
                      file_id='access_om3_cice_1day_mean_XXXX_XX').to_dask(
        xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
        ),
        xarray_open_kwargs = dict(
            chunks={"nj": -1, "ni": -1},
            decode_timedelta=True
        )
    )
    ds['time'] = ds.indexes['time'].normalize()
    ds_wav = datastore.search(variable=["HS", "ICEF", "ICEH"], frequency="fx").to_dask(
        xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
        ),
        xarray_open_kwargs = dict(
            chunks={"ny": -1, "nx": -1},
            decode_timedelta=True
        )
    )
    ds_wav = ds_wav.rename({'ny': 'nj',
                            'nx': 'ni'}
    )

    new_time = pd.date_range(
        start="2010-01-01",
        periods=ds_wav.sizes['time'],
        freq="D"
    )
    # new_time = ds.time.values
    ds_wav = ds_wav.assign_coords(time=new_time)


    ds_grid = datastore.search(variable=["tarea", "HTE", "NFSD"], frequency="fx", realm="seaIce").to_dask().compute()
    ds_grid
    
    coords = datastore.search(variable=["TLAT", "TLON"], #file_id='access_om3_mom6_static'
                             ).to_dask().compute() # TODO why do we need file_id for my runs??
    coords = coords.fillna(0.0)
    # coords = coords.rename({'geolat': 'lat', 'geolon': 'lon'})
    ds = xr.merge([ds, ds_wav])
    ds = xr.merge([ds, ds_grid])
    ds = ds.assign_coords(coords)
    ds = ds.roll(ni=80, roll_coords=True)
    # ds = ds.isel(time=slice(0,10*365))
    try:
        lon, lat = get_lon_lat_from_catalog(datastore)
        print("Discovered longitude/latitude from datastore catalog.")
    except RuntimeError as e:
        print(f"Catalog lon/lat lookup failed: {e}")
        lon, lat = _align_lonlat_dims(ds, lon, lat)
    lon = lon.roll(lonh=80, roll_coords=True)
    lon = lon.rename({'lonh': 'ni'})
    lon = lon % 360
    lat = lat.rename({'lath': 'nj'})
    ds = ds.cf.assign_coords(
            {"longitude": lon,
             "latitude": lat,
            }
    )

    return ds

ds_list = [read_in_data(ds) for ds in datastores.values()]


In [ ]:
ds_list[0]

In [ ]:
for name, datastore in datastores.items():
    print(f"\n{name}")
    print(datastore.search(variable="HS", frequency="fx"))

In [ ]:
# ds.search(variable=["HS"], frequency="fx")

### Align the three datasets

In [ ]:
ds_aligned = xr.align(*ds_list)

In [ ]:
ds_aligned[0].time

In [ ]:
# ds_align1.TLON.isel(nj=100).values

### Plot global `HS` and `sig_wav_ht`

In [ ]:
time_idx = -180
# ds_plots = [ds_align1, ds_align2, ds_align3, ds_align4]
ds_plots = ds_aligned #[ds_align1, ds_align2]
n_models = len(ds_plots)

In [ ]:
# create subplots with Robinson projection
fig, axes = plt.subplots(
    2, len(ds_plots), 
    figsize=(5*n_models, 5),
    subplot_kw={'projection': ccrs.Robinson(central_longitude=0)}
)

for i in range(n_models):
    # Top Row
    ax = axes[0,i]
    da = ds_plots[i]['HS'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['ni'], da['nj'], da,
        transform=ccrs.PlateCarree(),
        vmin=0, vmax=10, cmap='viridis'
    )

    ax.set_title(expt_names[i])

    # Bottom Row
    ax = axes[1,i]
    da = ds_plots[i]['wave_sig_ht'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['ni'], da['nj'], da,
        transform=ccrs.PlateCarree(),
        vmin=0, vmax=10, cmap='viridis'
    )
for ax in axes.flatten():
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.set_global()
    ax.coastlines()
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    
date = pd.to_datetime(ds_plots[0]['HS'].isel(time=time_idx)['time'].values)
fig.suptitle(date.strftime('%Y-%m-%d'), fontsize=14, y=0.985)

In [ ]:
# create subplots with SouthPolar projection
fig, axes = plt.subplots(
    2, len(ds_plots), 
    figsize=(4*n_models, 5),
    subplot_kw={'projection': ccrs.SouthPolarStereo()}
)
for i in range(n_models):
    # Top Row
    ax = axes[0,i]
    da = ds_plots[i]['HS'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['ni'], da['nj'], da,
        transform=ccrs.PlateCarree(),
        vmin=0, vmax=10, cmap='viridis'
    )

    ax.set_title(expt_names[i])

    # Bottom Row
    ax = axes[1,i]
    da = ds_plots[i]['wave_sig_ht'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['ni'], da['nj'], da,
        transform=ccrs.PlateCarree(),
        vmin=0, vmax=10, cmap='viridis'
    )

import matplotlib.path as mpath

theta = np.linspace(0, 2*np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)

for ax in axes.flatten():
    ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.coastlines()
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

cbar = fig.colorbar(
    img,
    ax=axes,
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar.set_label('Significant Wave Height (m)')

date = pd.to_datetime(ds_plots[0]['HS'].isel(time=time_idx)['time'].values)
fig.suptitle(date.strftime('%Y-%m-%d'), fontsize=14, y=0.985)

In [ ]:
# create subplots with SouthPolar projection and logscale
fig, axes = plt.subplots(
    2, len(ds_plots), 
    figsize=(4*n_models, 5),
    subplot_kw={'projection': ccrs.SouthPolarStereo()}
)
levels = [0.1, 1, 5]  # 1e-3 to 10
for i in range(n_models):
    # Top Row
    ax = axes[0,i]
    da = ds_plots[i]['HS'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        norm=LogNorm(vmin=1e-4, vmax=15)
    )
    da = ds_plots[i]['aice'].isel(time=time_idx)
    ax.contour(
        da['longitude'], da['latitude'],  da,
        levels=[0.15, 0.8],
        colors='white',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )

    ax.set_title(expt_names[i])

    # Bottom Row
    ax = axes[1,i]
    da = ds_plots[i]['wave_sig_ht'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        norm=LogNorm(vmin=1e-4, vmax=15)
    )
    da = ds_plots[i]['aice'].isel(time=time_idx)
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=[0.15, 0.8],
        colors='white',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )
for ax in axes.flatten():
    ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.coastlines()
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

cbar = fig.colorbar(
    img,
    ax=axes,
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar.set_label('Significant Wave Height (m)')

date = pd.to_datetime(ds_plots[0]['HS'].isel(time=time_idx)['time'].values)
fig.suptitle(date.strftime('%Y-%m-%d'), fontsize=14, y=0.985)

In [ ]:
# create subplots with SouthPolar projection and logscale
fig, axes = plt.subplots(
    2, len(ds_plots), 
    figsize=(4*n_models, 5),
    subplot_kw={'projection': ccrs.SouthPolarStereo()}
)
levels = [0.1, 1, 5]  # 1e-3 to 10
for i in range(n_models):
    # Top Row
    ax = axes[0,i]
    da = ds_plots[i]['ICEH'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        vmin=0,
        vmax=5,
    )
    da = ds_plots[i]['aice'].isel(time=time_idx)
    ax.contour(
        da['longitude'], da['latitude'],  da,
        levels=[0.15, 0.8],
        colors='white',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )

    ax.set_title(expt_names[i])

    # Bottom Row
    ax = axes[1,i]
    da = ds_plots[i]['hi'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        vmin=0,
        vmax=5,
    )
    da = ds_plots[i]['aice'].isel(time=time_idx)
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=[0.15, 0.8],
        colors='white',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )
for ax in axes.flatten():
    ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.coastlines()
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

cbar = fig.colorbar(
    img,
    ax=axes,
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar.set_label('Mean ice thickness (m)')

date = pd.to_datetime(ds_plots[0]['HS'].isel(time=time_idx)['time'].values)
fig.suptitle(date.strftime('%Y-%m-%d'), fontsize=14, y=0.985)

In [ ]:
# create subplots with SouthPolar projection and logscale
fig, axes = plt.subplots(
    2, len(ds_plots), 
    figsize=(4*n_models, 5),
    subplot_kw={'projection': ccrs.SouthPolarStereo()}
)
vmin, vmax = 0, 1000
for i in range(n_models):
    # Top Row
    ax = axes[0,i]
    da = ds_plots[i]['ICEF'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['longitude'], da['latitude'], da/2,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        vmin=vmin,
        vmax=vmax,
    )
    da = ds_plots[i]['aice'].isel(time=time_idx)
    ax.contour(
        da['longitude'], da['latitude'],  da,
        levels=[0.15, 0.8],
        colors='white',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )

    ax.set_title(expt_names[i])

    # Bottom Row
    ax = axes[1,i]
    da = ds_plots[i]['fsdrad'].isel(time=time_idx)
    img = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        vmin=vmin,
        vmax=vmax,
    )
    da = ds_plots[i]['aice'].isel(time=time_idx)
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=[0.15, 0.8],
        colors='white',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )
for ax in axes.flatten():
    ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.coastlines()
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

cbar = fig.colorbar(
    img,
    ax=axes,
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar.set_label('Representative floe size (m)')

date = pd.to_datetime(ds_plots[0]['HS'].isel(time=time_idx)['time'].values)
fig.suptitle(date.strftime('%Y-%m-%d'), fontsize=14, y=0.985)

In [ ]:
# create subplots with SouthPolar projection and logscale
fig, axes = plt.subplots(
    2, len(ds_plots), 
    figsize=(4*n_models, 5),
    subplot_kw={'projection': ccrs.SouthPolarStereo()}
)
levels = [0.15, 0.8]  # 1e-3 to 10
for i in range(n_models):
    # Top Row
    ax = axes[0,i]
    da = ds_plots[i]['aice'].isel(time=time_idx)
    img_top = ax.pcolormesh(
        da['ni'], da['nj'], da,
        transform=ccrs.PlateCarree(),
        cmap=cmo.ice,
        vmin=0,
        vmax=1,
    )
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=levels,
        colors='white',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )

    ax.set_title(expt_names[i])

    # Bottom Row
    ax = axes[1,i]
    da = ds_plots[i]['fsdrad'].isel(time=time_idx)
    img_bottom = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        norm=LogNorm(vmin=1, vmax=850)
    )
    da = ds_plots[i]['aice'].isel(time=time_idx)
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=levels,
        colors='white',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )
for ax in axes.flatten():
    ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.coastlines()
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

# Top row colorbar
cbar_top = fig.colorbar(
    img_top,
    ax=axes[0, :],
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar_top.set_label('Sea ice concentration')

# Bottom row colorbar
cbar_bottom = fig.colorbar(
    img_bottom,
    ax=axes[1, :],
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar_bottom.set_label('Rep. floe size (m)')

date = pd.to_datetime(ds_plots[0]['HS'].isel(time=time_idx)['time'].values)
fig.suptitle(date.strftime('%Y-%m-%d'), fontsize=14, y=0.985)

In [ ]:
# Import my functions
functions_path = os.path.abspath("/home/566/nd0349/access-om3-analysis/functions")
if functions_path not in sys.path:
    sys.path.append(functions_path)
from get_files import *
from plot_settings import *
from fstd import *
from attenuation_models import *
test()

def integrateFloeSize(ds, var):
    puny = 1e-12
    tmp_var = var[0:-3]

    # Mask once
    ds_masked = ds.where(ds['aice'] > puny)

    # Get weights
    NFSD = xr.DataArray(ds.NFSD.values, dims=["nf"])

    # Weighted sum over nf
    int_var = (ds_masked[tmp_var] * NFSD).sum(dim="nf") / ds_masked["aice"]

    # Fill NaNs where aice is tiny (optional)
    int_var = int_var.fillna(0)

    # Assign to output dataset
    ds_out = ds.copy()
    ds_out[var] = int_var

    # Attributes
    parts = ds[tmp_var].attrs['long_name'].split(':')
    ds_out[var].attrs['long_name'] = parts[0][:-3] + 'rep. radius:' + parts[1]
    ds_out[f'{var}'].attrs['units'] = 'm/timestep'

    return ds_out
    
# ds1 = integrateFloeSize(ds1, 'dafsd_wave_ra')
# ds2 = integrateFloeSize(ds2, 'dafsd_wave_ra')

ds_fsd = [integrateFloeSize(ds, 'dafsd_wave_ra') for ds in ds_plots]
ds_fsd = [integrateFloeSize(ds, 'dafsd_weld_ra') for ds in ds_fsd]

In [ ]:
ds_plots = ds_fsd
ds_plots[0]

In [ ]:
# time_idx = -1
# create subplots with SouthPolar projection and logscale
fig, axes = plt.subplots(
    3, len(ds_plots), 
    figsize=(4*n_models, 10),
    subplot_kw={'projection': ccrs.SouthPolarStereo()}
)

da = ds_plots[i]['dafsd_wave_ra'].isel(time=time_idx, nj=slice(0,150))
vmin, vmax = da.min().values/2, abs(da.min().values)/2

for i in range(n_models):
    # Top Row
    ax = axes[0,i]
    da = ds_plots[i]['fsdrad'].isel(time=time_idx)
    img_top = ax.pcolormesh(
        da['ni'], da['nj'], da,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        norm=LogNorm(vmin=1, vmax=850)
    )
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=[0.15, 0.8],
        colors='magenta',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )

    ax.set_title(expt_names[i])

    # Middle Row
    ax = axes[1,i]
    da = ds_plots[i]['dafsd_wave_ra'].isel(time=time_idx)
    img_middle = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap=cmo.balance,
        vmin=vmin/10,
        vmax=vmax/10,
    )
    da = ds_plots[i]['aice'].isel(time=time_idx)
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=[0.15, 0.8],
        colors='k',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )

    # Bottom Row
    ax = axes[2,i]
    da = ds_plots[i]['dafsd_weld_ra'].isel(time=time_idx)
    img_bottom = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap=cmo.balance,
        vmin=vmin,
        vmax=vmax,
    )
    da = ds_plots[i]['aice'].isel(time=time_idx)
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=[0.15, 0.8],
        colors='k',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )
for ax in axes.flatten():
    ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.coastlines()
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

# Top row colorbar
cbar_top = fig.colorbar(
    img_top,
    ax=axes[0, :],
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar_top.set_label('Rep. floe size [m]')

# Middle row colorbar
cbar_middle = fig.colorbar(
    img_middle,
    ax=axes[1, :],
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar_middle.set_label('Change in floe size from waves [m/h]')
# Bottom row colorbar
cbar_bottom = fig.colorbar(
    img_bottom,
    ax=axes[2, :],
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar_bottom.set_label('Change in floe size from welding [m/h]')

date = pd.to_datetime(ds_plots[0]['aice'].isel(time=time_idx)['time'].values)
fig.suptitle(date.strftime('%Y-%m-%d'), fontsize=14, y=0.985)

In [ ]:
# time_idx = -1
# create subplots with SouthPolar projection and logscale
fig, axes = plt.subplots(
    3, len(ds_plots), 
    figsize=(4*n_models, 10),
    subplot_kw={'projection': ccrs.SouthPolarStereo()}
)

da = ds_plots[i]['dafsd_wave_ra'].isel(time=time_idx, nj=slice(0,150))
vmin, vmax = da.min().values/2, abs(da.min().values)/2

for i in range(n_models):
    # Top Row
    ax = axes[0,i]
    da = ds_plots[i]['fsdrad'].mean(dim='time')
    img_top = ax.pcolormesh(
        da['ni'], da['nj'], da,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        norm=LogNorm(vmin=1, vmax=850)
    )
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=[0.15, 0.8],
        colors='magenta',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )

    ax.set_title(expt_names[i])

    # Middle Row
    ax = axes[1,i]
    da = ds_plots[i]['dafsd_wave_ra'].mean(dim='time')
    img_middle = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap=cmo.balance,
        vmin=vmin/10,
        vmax=vmax/10,
    )
    da = ds_plots[i]['aice'].mean(dim='time')
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=[0.15, 0.8],
        colors='k',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )

    # Bottom Row
    ax = axes[2,i]
    da = ds_plots[i]['dafsd_weld_ra'].mean(dim='time')
    img_bottom = ax.pcolormesh(
        da['longitude'], da['latitude'], da,
        transform=ccrs.PlateCarree(),
        cmap=cmo.balance,
        vmin=vmin,
        vmax=vmax,
    )
    da = ds_plots[i]['aice'].mean(dim='time')
    ax.contour(
        da['longitude'], da['latitude'], da,
        levels=[0.15, 0.8],
        colors='k',
        linewidths=0.6,
        transform=ccrs.PlateCarree()
    )
for ax in axes.flatten():
    ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.coastlines()
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

# Top row colorbar
cbar_top = fig.colorbar(
    img_top,
    ax=axes[0, :],
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar_top.set_label('Rep. floe size [m]')

# Middle row colorbar
cbar_middle = fig.colorbar(
    img_middle,
    ax=axes[1, :],
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar_middle.set_label('Change in floe size from waves [m/h]')
# Bottom row colorbar
cbar_bottom = fig.colorbar(
    img_bottom,
    ax=axes[2, :],
    orientation='vertical',
    fraction=0.02,
    pad=0.04
)
cbar_bottom.set_label('Change in floe size from welding [m/h]')

date = pd.to_datetime(ds_plots[0]['aice'].isel(time=time_idx)['time'].values)
# fig.suptitle(date.strftime('%Y-%m-%d'), fontsize=14, y=0.985)